# Project 5 — Advanced solution: an enum-backed exception catalog

**Scope:** Solve the supplied project completely while retaining the public API `AppException.Timeout.throw()` and `AppException.NotAnInteger.throw(...)`. Standard-library-only; suitable for contemporary Python 3 (tested in Python 3.13). No restriction to the 3.6 version in the source notebook.

**Specification clarification:** The introductory example says timeout code 100 and a `ConnectionException`, but the source's final working enum actually uses `Generic=100`, `Timeout=101` and an exception class named `Timeout`. This solution follows the final implementation: codes **100, 101, 200, 201**. The error texts also follow the working code, rather than the slightly different introductory prose.

**Design:** explicit `Enum.__new__`, unique codes, typed exception classes, readable immutable enum metadata, error construction separate from raising, name/code lookup, documentation-friendly catalog, and deterministic tests. We validate inputs rather than silently interpreting `''` as an omitted override; `None` means use the default. No external libraries are required.

## 1. Domain exception classes

Custom exception types permit targeted `except` clauses. `Timeout` inherits from `GenericException` so callers can catch either the specific issue or all catalog-defined connection-like exceptions in this small example. Built-in `ValueError` remains the type for validation errors, as the source requires.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum, unique
from typing import NoReturn


class GenericException(Exception):
    """Base exception for custom catalog errors."""


class Timeout(GenericException):
    """Raised when an application resource does not respond in time."""


@dataclass(frozen=True, slots=True)
class ErrorDefinition:
    """A read-only, documentation-friendly view of one catalog member."""

    name: str
    code: int
    exception_type: type[Exception]
    message: str

## 2. Implement the enumeration

`Enum.value` is the unique integer error code, while each member also has its exception class and default message. `@unique` prevents accidental duplicate integer codes. `make_exception` supports constructing an exception for explicit `raise`, tests, or integration with other frameworks; `throw` is the requested concise raising method.

In [2]:
@unique
class AppException(Enum):
    """One central source of truth for application exception definitions."""

    Generic = (100, GenericException, "Application exception.")
    Timeout = (101, Timeout, "Timeout connecting to resource.")
    NotAnInteger = (200, ValueError, "Value must be an integer.")
    NotAList = (201, ValueError, "Value must be a list.")

    def __new__(cls, code: int, exception_type: type[Exception], message: str):
        if type(code) is not int or code < 0:
            raise ValueError("Error codes must be non-negative integers (not bool).")
        if not isinstance(exception_type, type) or not issubclass(exception_type, Exception):
            raise TypeError("exception_type must be an Exception subclass.")
        if not isinstance(message, str) or not message.strip():
            raise ValueError("Default error messages must be nonempty strings.")

        member = object.__new__(cls)
        member._value_ = code
        member.exception = exception_type
        member.message = message
        return member

    @property
    def code(self) -> int:
        """Stable numeric error code, also available through ``value``."""
        return self.value

    def make_exception(self, message: str | None = None) -> Exception:
        """Construct, but do not raise, a correctly typed and formatted error."""
        if message is None:
            message = self.message
        if not isinstance(message, str):
            raise TypeError("An override message must be a string or None.")
        if not message.strip():
            raise ValueError("An override message cannot be blank.")
        return self.exception(f"{self.code} - {message}")

    def throw(self, message: str | None = None) -> NoReturn:
        """Raise the catalog's configured exception, optionally overriding its text."""
        raise self.make_exception(message)

    @classmethod
    def catalog(cls) -> tuple[ErrorDefinition, ...]:
        """Return a snapshot of all documented error definitions, in declaration order."""
        return tuple(
            ErrorDefinition(item.name, item.code, item.exception, item.message)
            for item in cls
        )

## 3. Usage and exception handling

Do not let deliberate exception demonstrations interrupt the notebook: use `try`/`except`. Lookup by code (`AppException(101)`) and by key (`AppException['Timeout']`) are native `Enum` capabilities.

In [3]:
print("Lookup by name:", AppException["Timeout"])
print("Lookup by code:", AppException(101))
print("Timeout metadata:", AppException.Timeout.code, AppException.Timeout.exception.__name__,
      AppException.Timeout.message)

for item, custom_text in [
    (AppException.Timeout, None),
    (AppException.Timeout, "Timeout connecting to database."),
    (AppException.NotAnInteger, None),
]:
    try:
        item.throw(custom_text)
    except (GenericException, ValueError) as exc:
        print(f"Caught {type(exc).__name__}: {exc}")

print("All registered errors:")
for definition in AppException.catalog():
    print(f"  {definition.name:14} {definition.code:3} "
          f"{definition.exception_type.__name__:18} {definition.message}")

Lookup by name: AppException.Timeout
Lookup by code: AppException.Timeout
Timeout metadata: 101 Timeout Timeout connecting to resource.
Caught Timeout: 101 - Timeout connecting to resource.
Caught Timeout: 101 - Timeout connecting to database.
Caught ValueError: 200 - Value must be an integer.
All registered errors:
  Generic        100 GenericException   Application exception.
  Timeout        101 Timeout            Timeout connecting to resource.
  NotAnInteger   200 ValueError         Value must be an integer.
  NotAList       201 ValueError         Value must be a list.


## 4. Unit tests

Tests cover exact text and exception types, enum lookup, override semantics, code uniqueness, metadata, catalog invariants, immutability, and explicit construction. `unittest` runs without third-party dependencies. A failing suite raises an `AssertionError`, making automated notebook execution fail immediately.

In [4]:
import unittest
from dataclasses import FrozenInstanceError


class TestAdvancedAppException(unittest.TestCase):
    def test_catalog_has_exact_four_keys(self):
        self.assertEqual([e.name for e in AppException],
                         ["Generic", "Timeout", "NotAnInteger", "NotAList"])

    def test_codes_are_stable(self):
        self.assertEqual([e.code for e in AppException], [100, 101, 200, 201])

    def test_unique_codes(self):
        self.assertEqual(len({item.value for item in AppException}), len(AppException))

    def test_enum_lookup_by_name(self):
        self.assertIs(AppException["NotAList"], AppException.NotAList)

    def test_enum_lookup_by_code(self):
        self.assertIs(AppException(101), AppException.Timeout)

    def test_invalid_lookup_by_code(self):
        with self.assertRaises(ValueError):
            AppException(999)

    def test_invalid_lookup_by_name(self):
        with self.assertRaises(KeyError):
            AppException["Missing"]

    def test_code_property_matches_value(self):
        for item in AppException:
            with self.subTest(item=item):
                self.assertEqual(item.code, item.value)

    def test_exception_classes(self):
        self.assertIs(AppException.Generic.exception, GenericException)
        self.assertIs(AppException.Timeout.exception, Timeout)
        self.assertIs(AppException.NotAnInteger.exception, ValueError)
        self.assertIs(AppException.NotAList.exception, ValueError)

    def test_timeout_is_generic_subclass(self):
        self.assertTrue(issubclass(Timeout, GenericException))

    def test_default_messages(self):
        self.assertEqual(AppException.Timeout.message, "Timeout connecting to resource.")
        self.assertEqual(AppException.NotAnInteger.message, "Value must be an integer.")

    def test_throw_timeout_type_and_text(self):
        with self.assertRaisesRegex(Timeout, r"^101 - Timeout connecting to resource\.$"):
            AppException.Timeout.throw()

    def test_throw_integer_type_and_text(self):
        with self.assertRaisesRegex(ValueError, r"^200 - Value must be an integer\.$"):
            AppException.NotAnInteger.throw()

    def test_throw_list_type_and_text(self):
        with self.assertRaisesRegex(ValueError, r"^201 - Value must be a list\.$"):
            AppException.NotAList.throw()

    def test_throw_generic_type_and_text(self):
        with self.assertRaisesRegex(GenericException, r"^100 - Application exception\.$"):
            AppException.Generic.throw()

    def test_override_message(self):
        with self.assertRaisesRegex(Timeout, r"^101 - Database unavailable$"):
            AppException.Timeout.throw("Database unavailable")

    def test_overrides_do_not_mutate_defaults(self):
        with self.assertRaises(Timeout):
            AppException.Timeout.throw("Alternate")
        self.assertEqual(AppException.Timeout.message, "Timeout connecting to resource.")

    def test_override_none_means_default(self):
        self.assertEqual(str(AppException.Timeout.make_exception(None)),
                         "101 - Timeout connecting to resource.")

    def test_bad_override_type(self):
        with self.assertRaises(TypeError):
            AppException.Timeout.make_exception(42)

    def test_blank_override_rejected(self):
        for message in ("", "   ", "\n"):
            with self.subTest(message=message), self.assertRaises(ValueError):
                AppException.Timeout.make_exception(message)

    def test_make_exception_does_not_raise(self):
        err = AppException.Timeout.make_exception("Other")
        self.assertIsInstance(err, Timeout)
        self.assertEqual(str(err), "101 - Other")

    def test_catalog_is_tuple(self):
        self.assertIsInstance(AppException.catalog(), tuple)

    def test_catalog_matches_enum(self):
        for definition, item in zip(AppException.catalog(), AppException, strict=True):
            self.assertEqual((definition.name, definition.code,
                              definition.exception_type, definition.message),
                             (item.name, item.code, item.exception, item.message))

    def test_catalog_rows_are_frozen(self):
        with self.assertRaises(FrozenInstanceError):
            AppException.catalog()[0].code = 900

    def test_invalid_code_constructor(self):
        # __new__ validation can be exercised with a separate Enum declaration.
        with self.assertRaises(ValueError):
            class Bad(Enum):
                def __new__(cls, code, exc_type, message):
                    return AppException._new_member_(cls, code, exc_type, message)
                Example = (True, ValueError, "Message")

    def test_invalid_exception_constructor(self):
        with self.assertRaises(TypeError):
            class Bad(Enum):
                def __new__(cls, code, exc_type, message):
                    return AppException._new_member_(cls, code, exc_type, message)
                Example = (3, int, "Message")

    def test_invalid_message_constructor(self):
        with self.assertRaises(ValueError):
            class Bad(Enum):
                def __new__(cls, code, exc_type, message):
                    return AppException._new_member_(cls, code, exc_type, message)
                Example = (3, ValueError, "  ")


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestAdvancedAppException)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), "Advanced notebook tests failed"
print(f"ALL TESTS PASSED (advanced: {result.testsRun})")

...........................
----------------------------------------------------------------------
Ran 27 tests in 0.027s

OK


ALL TESTS PASSED (advanced: 27)


## 5. Design notes and trade-offs

- **Why an enum?** It gives one discoverable, uniquely-coded catalog and provides native lookups by name and code. Error keys and codes are part of the application contract and should not be renumbered casually.
- **Why `Exception`, not `BaseException`?** `KeyboardInterrupt` and `SystemExit` inherit from `BaseException`; they should not be disguised as application errors.
- **Why validate `message is None` explicitly?** `message or self.message` silently discards an intentionally supplied empty string. This design rejects blanks explicitly.
- **Why keep built-in `ValueError`?** It is the observed behavior of the supplied final notebook. The extended solution adds a coherent custom error hierarchy and structured context.
- **Production boundary:** An in-memory catalog does not persist telemetry, handle localization, or provide a cross-process error registry. Document codes and make changes deliberately. Never embed secrets in user-facing exception messages.